# 📥 Dataset Access Hub — all 24 registered benchmarks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/AnyDisasterMapping/blob/main/%URL%)
[![GitHub](https://img.shields.io/badge/Repo-AnyDisasterMapping-181717?logo=github)](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping)
![Auto](https://img.shields.io/badge/runs-automatically-3fb950)
![Scope](https://img.shields.io/badge/datasets-24--registered-blue) ![Task](https://img.shields.io/badge/task-seg%20%2B%20cd%20%2B%20scd-4c8cbf)
One-stop access manager for every dataset registered in the framework: an **access card** per dataset (where to request it, which adapter loads it), three fetch strategies (automated pull · your own download link · zip upload), and a **structure verifier** that previews tiles and checks train/val/test layout. Never blocks: if no data is present, a labeled synthetic sample keeps the run alive.


In [ ]:
#@title ⚙️ SETUP — auto-installs, idempotent {display-mode:"form"}
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
_pip("prettytable", "pillow", "numpy", "imageio", "scikit-image", "tifffile", "matplotlib", "gdown", "huggingface_hub", "ipywidgets")
import os, random
import numpy as np
random.seed(42); np.random.seed(42)
NOTEBOOK = "dataset_download_colab.ipynb"
STATE = {}
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
print("✅ setup complete")

In [ ]:
#@title ☁️ MOUNT GOOGLE DRIVE — persistent uploads, checkpoints & results {display-mode:"form"}
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT = "AnyDisasterMapping"  #@param {type:"string"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

DRIVE_ROOT, IN_COLAB = colab_kit.mount_and_prepare(DRIVE_PROJECT, MOUNT_DRIVE)

In [ ]:
#@title 🗂️ CATALOG — the 24 registered benchmarks {display-mode:"form"}
#@markdown Pick a dataset to get its access card. **Access types:** `auto` = downloadable in this notebook · `registration` = request from the data owner (link below) · `upload` = you already have the zip.

import json, os
from prettytable import PrettyTable

DATASETS = [
 # key, name, domain, task, sensor, access, host / where to get it, adapter in repo
 ("xbd",            "xBD (xView2)",            "infra damage", "cd",  "optical",        "registration", "xview2.org (requested access)",   "src/datasets/infra_xbd.py"),
 ("bright",         "BRIGHT",                  "infra damage", "cd",  "optical+SAR",    "registration", "github.com/SyedRafiAhmedGIS/BRIGHT", "src/datasets/infra_bright.py"),
 ("rescuenet",      "RescueNet",               "infra damage", "seg", "optical UAV",    "registration", "IEEE DataPort",                    "src/datasets/infra_rescuenet.py"),
 ("second",         "SECOND",                  "land cover",   "scd", "optical",        "registration", "GitHub (SECOND authors)",          "src/datasets/second.py"),
 ("cau_flood",      "CAU Flood",               "flood",        "cd",  "optical",        "registration", "GitHub (CAU authors)",             "src/datasets/flood_cau.py"),
 ("kurosiwo",       "KuroSiwo",                "flood",        "seg/cd","SAR",          "registration", "IEEE DataPort",                    "src/datasets/flood_kurosiwo.py"),
 ("urbansar",       "UrbanSARFloods",          "flood",        "seg/cd","SAR",          "huggingface",  "Hugging Face datasets",            "src/datasets/flood_urbansar.py"),
 ("gvlm",           "GVLM",                    "landslide",    "cd",  "optical",        "registration", "GitHub (GVLM authors)",            "src/datasets/landslide_gvlm.py"),
 ("l4s",            "Landslide4Sense",         "landslide",    "seg", "multi-spectral", "huggingface",  "Hugging Face competition",         "src/datasets/landslide_l4s.py"),
 ("hrgldd",         "HRGLDD",                  "landslide",    "seg", "multi-spectral", "registration", "Zenodo",                           "src/datasets/landslide_hrgldd.py"),
 ("s2wcd",          "S2WCD",                   "wildfire",     "cd",  "Sentinel-2",     "registration", "GitHub (S2WCD authors)",           "src/datasets/wildfire_s2wcd.py"),
 ("floga",          "FLOGA",                   "wildfire",     "seg", "Sentinel-2",     "registration", "Zenodo / IEEE DataPort",           "src/datasets/wildfire_floga.py"),
 ("satellite_burned_area", "Satellite Burned Area", "wildfire","seg","Sentinel-2",    "registration", "GitHub (burned-area authors)",     "src/datasets/wildfire_satba.py"),
 ("fire_spread",    "Next-Day Wildfire Spread","wildfire",     "seg", "multi-spectral", "direct",       "Google Research (TFDS mirror)",    "src/datasets/wildfire_firespread.py"),
 ("oilspill_sar",   "Oil Spill SAR",           "oil spill",    "seg", "SAR",            "registration", "see paper §extended domains",      "src/datasets/oilspill_sar.py"),
 ("avalanche_slf",  "Avalanche SLF",           "avalanche",    "seg", "optical",        "registration", "WSL SLF institute",                "src/datasets/avalanche_slf.py"),
 ("volcano_pyraws", "PyRAWs",                  "volcano",      "seg", "optical+SAR",    "registration", "GitHub (PyRAWs authors)",          "src/datasets/volcano_pyraws.py"),
 ("tsunami_inundation","Tsunami Inundation",   "tsunami",      "cd",  "optical",        "registration", "see paper §extended domains",      "src/datasets/tsunami_inundation.py"),
 ("hurricane_surge","S1 Storm Surge",          "hurricane",    "cd",  "SAR",            "registration", "see paper §extended domains",      "src/datasets/hurricane_surge.py"),
 ("glof_icimod",    "GLOF ICIMOD",             "glof",         "seg/scd","optical",     "registration", "ICIMOD",                           "src/datasets/glof_icimod.py"),
 ("duststorm_modis","MODIS Dust Storm",        "dust storm",   "seg", "MODIS",          "registration", "see paper §extended domains",      "src/datasets/duststorm_modis.py"),
 ("smoke_plume",    "Smoke Plume MODIS",       "smoke plume",  "seg", "MODIS",          "registration", "see paper §extended domains",      "src/datasets/smoke_plume.py"),
 ("locust_habitat", "Locust Habitat",          "locust",       "seg", "optical",        "registration", "FAO / paper §extended domains",    "src/datasets/locust_habitat.py"),
]
BY_KEY = {d[0]: d for d in DATASETS}

DATASET = "urbansar"  #@param ["xbd","bright","rescuenet","second","cau_flood","kurosiwo","urbansar","gvlm","l4s","hrgldd","s2wcd","floga","satellite_burned_area","fire_spread","oilspill_sar","avalanche_slf","volcano_pyraws","tsunami_inundation","hurricane_surge","glof_icimod","duststorm_modis","smoke_plume","locust_habitat"]

t = PrettyTable()
t.field_names = ["key", "dataset", "domain", "task", "sensor", "access"]
t.align = "l"
for d in DATASETS:
    t.add_row(d[:6])
print(t.get_string())

k, name, domain, task, sensor, access, host, adapter = BY_KEY[DATASET]
W = 66
print("\n┌" + "─" * W + "┐")
print("│ 📇 ACCESS CARD — " + name.ljust(W - 19) + "│")
print("├" + "─" * W + "┤")
for line in [f"domain     : {domain}", f"task       : {task}", f"sensor     : {sensor}",
             f"get it from: {host}", f"adapter    : {adapter}"]:
    print("│ " + line.ljust(W - 2) + "│")
print("└" + "─" * W + "┘")
STATE = {"dataset": k, "name": name, "access": access, "adapter": adapter}
if access == "registration":
    print("\n📌 This dataset needs a registration/request step — the owner link")
    print("   is in the card above. Once approved, upload the zip in the next")
    print("   cell (or paste your personal download link / Drive file id).")
elif access == "huggingface":
    print("\n📌 Public on Hugging Face — the FETCH cell will try an automated")
    print("   pull with huggingface_hub (no token needed for public repos).")
else:
    print("\n📌 Direct access — FETCH cell will attempt an automated download;")
    print("   if the mirror moved, use the zip-upload path instead.")


In [ ]:
#@title 📥 FETCH — auto-pull · zip upload · own download link (never blocks) {display-mode:"form"}
TARGET_DIR = f"data/{STATE['dataset']}"  #@param {type:"string"}
GDRIVE_FILE_ID = ""  #@param {type:"string"}
UPLOAD_ZIP_IF_NO_AUTO = True  #@param {type:"boolean"}

import os, shutil, zipfile as zf

os.makedirs(TARGET_DIR, exist_ok=True)
FETCHED = False

def extract_zip_bytes(data: bytes):
    import io
    zf.ZipFile(io.BytesIO(data)).extractall(TARGET_DIR)
    print("🧩 extracted zip →", TARGET_DIR)

# -- strategy 1: Hugging Face public dataset ---------------------------------
if STATE["access"] == "huggingface":
    try:
        from huggingface_hub import snapshot_download
        p = snapshot_download(repo_id=f"AnyDisasterMapping/{STATE['dataset']}",
                              repo_type="dataset", local_dir=TARGET_DIR)
        print("✅ downloaded from Hugging Face →", p)
        FETCHED = True
    except Exception as e:
        print("ℹ️ automated HF pull unavailable:", str(e)[:120])
        print("   (weights/data mirror may not be live yet — use upload path below)")

# -- strategy 2: user-provided Drive file id ---------------------------------
elif GDRIVE_FILE_ID:
    try:
        import gdown
        out = os.path.join("/content", f"{STATE['dataset']}_download.zip")
        gdown.download(id=GDRIVE_FILE_ID, output=out, quiet=False)
        extract_zip_bytes(open(out, "rb").read())
        FETCHED = True
    except Exception as e:
        print("ℹ️ gdown failed:", str(e)[:120])

# -- strategy 3: manual zip upload -------------------------------------------
if not FETCHED and UPLOAD_ZIP_IF_NO_AUTO:
    print("📂 opening file picker — choose your dataset zip (any structure; the")
    print("   repo adapters in", STATE["adapter"], "document the expected layout)")
    try:
        from google.colab import files as _f
        up = _f.upload()
        for nm, data in up.items():
            if nm.lower().endswith(".zip"):
                extract_zip_bytes(data)
            else:
                with open(os.path.join(TARGET_DIR, nm), "wb") as fh:
                    fh.write(data)
        FETCHED = True
    except Exception as e:
        print("upload skipped:", str(e)[:100])

STATE.update(fetched=FETCHED, target_dir=TARGET_DIR)
print("\n→ data in:", TARGET_DIR, "| fetched:", FETCHED,
      "| files:", sum(len(fs) for _, _, fs in os.walk(TARGET_DIR)))


In [ ]:
#@title 🧪 VERIFY — structure check + 3×3 preview grid (synthetic fallback) {display-mode:"form"}
import glob, math, random, os
import numpy as np
import matplotlib.pyplot as plt

IMG_EXT = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".npy")
files = [p for ext in IMG_EXT for p in glob.glob(os.path.join(TARGET_DIR, "**", f"*{ext}"), recursive=True)]
print(f"🔎 found {len(files)} image/mask files under {TARGET_DIR}")

def read_any(p):
    if p.endswith(".npy"):
        return np.load(p, allow_pickle=True)
    import tifffile
    if p.endswith((".tif", ".tiff")):
        return tifffile.imread(p)
    import imageio.v2 as imageio
    return imageio.imread(p)

if len(files) >= 3:
    sample = random.Random(42).sample(files, min(9, len(files)))
    fig, axes = plt.subplots(3, 3, figsize=(9, 9), constrained_layout=True)
    for ax, p in zip(axes.ravel(), sample):
        try:
            arr = read_any(p)
            if arr.ndim == 3 and arr.shape[0] <= 16 and arr.shape[0] < arr.shape[-1]:  # CHW
                arr = arr[:3].transpose(1, 2, 0)
            if arr.ndim == 3 and arr.shape[-1] > 3:
                arr = arr[..., :3]
            lo, hi = np.percentile(arr, (2, 98))
            ax.imshow(np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1))
        except Exception as e:
            ax.text(.5, .5, f"preview n/a\n{str(e)[:40]}", ha="center", va="center")
        ax.set_title(os.path.basename(p)[:38], fontsize=8)
        ax.axis("off")
    fig.suptitle(f"{STATE['name']} — sample preview")
    plt.show()
    splits = {d for d in os.listdir(TARGET_DIR) if os.path.isdir(os.path.join(TARGET_DIR, d))}
    print("📁 top-level dirs (splits expected: train/val/test):", splits or "(flat layout)")
else:
    print("🛟 no data yet — generating a labeled SYNTHETIC sample so the pipeline")
    print("   stays runnable end-to-end (real data can be added anytime).")
    import numpy as np
    os.makedirs(TARGET_DIR, exist_ok=True)
    rng = np.random.default_rng(7)
    img = (rng.random((512, 512, 3)) * 0.4 + 0.3).astype(np.float32)
    yy, xx = np.mgrid[0:512, 0:512]
    blob = ((yy - 256) ** 2 + (xx - 300) ** 2) < 120 ** 2
    img[blob] *= 1.4
    mask = (blob * 1).astype(np.uint8)
    import imageio.v2 as imageio
    imageio.imwrite(os.path.join(TARGET_DIR, "synthetic_image.png"), (img * 255).astype(np.uint8))
    imageio.imwrite(os.path.join(TARGET_DIR, "synthetic_mask.png"), mask * 255)
    print("🛟 synthetic image+mask written to", TARGET_DIR)
    STATE["source"] = "SYNTHETIC-DEMO"
plt.close("all")


In [ ]:
#@title 📦 EXPORT — results zip + Drive copy + auto-download {display-mode:"form"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

def run_export():
    return colab_kit.export_results_zip_hub(globals())

run_export()

## 🗺️ The full notebook collection

| notebook | what it does |
|---|---|
| [`quickstart_colab.ipynb`](quickstart_colab.ipynb) | 5-minute tour of all 13 disaster domains |
| [`dataset_download_colab.ipynb`](dataset_download_colab.ipynb) | access hub for the 24 registered benchmarks |
| [`checkpoint_zoo_colab.ipynb`](checkpoint_zoo_colab.ipynb) | real weights → real inference (or train tiny in-session) |
| [`demo_colab.ipynb`](demo_colab.ipynb) | single-image demo across tasks |
| [`change_detection_colab.ipynb`](change_detection_colab.ipynb) | binary change detection (pre/post) |
| **`scd_colab.ipynb`** | **semantic change detection — land-cover *from→to* matrices** |
| **`evaluation_metrics_colab.ipynb`** | **mIoU · F1 · confusion matrices for any prediction** |
| [`satellite_finetune_colab.ipynb`](satellite_finetune_colab.ipynb) | finetune on your own labeled pairs |
| [`batch_inference_colab.ipynb`](batch_inference_colab.ipynb) | folder-level batch inference |
| domain notebooks (flood · wildfire · tsunami · hurricane · landslide · volcano) | per-domain guided demos |

Framework: [`SyedRafiAhmedGIS/AnyDisasterMapping`](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping) — paper: *Earth Observation for Disaster Mapping: Benchmarks, Methods, Challenges and Future Perspectives* (SSRN 6725082).
